In [ ]:
%pip install -q mediapipe

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%pip install mediapipe opencv-python

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#===================imports=======================
import cv2  # OpenCV for video processing
import mediapipe as mp # MediaPipe for hand tracking
import numpy as np # NumPy for numerical operations
import time # Time module for measuring FPS

from mediapipe.tasks import python  # Importing the Python API for MediaPipe tasks
from mediapipe.tasks.python import vision # Importing the vision module from MediaPipe tasks

In [ ]:
# ================== MODEL PATHS ==================
POSE_MODEL_PATH = "pose_landmarker_full.task" # Path to the pose landmarker model
HAND_MODEL_PATH = "hand_landmarker.task" # Path to the hand landmarker model

In [ ]:
# ================== BASE OPTIONS ==================
BaseOptions = python.BaseOptions # Base options for configuring the MediaPipe tasks
VisionRunningMode = vision.RunningMode # Running mode for the vision tasks (e.g., video, image, live stream)

In [ ]:
# ================== POSE SETUP ==================
PoseLandmarker = vision.PoseLandmarker # PoseLandmarker class for pose estimation
PoseLandmarkerOptions = vision.PoseLandmarkerOptions # Options for configuring the PoseLandmarker

pose_options = PoseLandmarkerOptions( # Configuring the options for the PoseLandmarker
    base_options=BaseOptions(model_asset_path=POSE_MODEL_PATH), # Setting the model path for the pose landmarker
    running_mode=VisionRunningMode.VIDEO, # Setting the running mode to video for frame-by-frame processing
    num_poses=1 # Setting the number of poses to detect (1 in this case)
)

In [ ]:
# ================== HAND SETUP ==================
HandLandmarker = vision.HandLandmarker # HandLandmarker class for hand tracking
HandLandmarkerOptions = vision.HandLandmarkerOptions # Options for configuring the HandLandmarker

hand_options = HandLandmarkerOptions( # Configuring the options for the HandLandmarker
    base_options=BaseOptions(model_asset_path=HAND_MODEL_PATH), # Setting the model path for the hand landmarker
    running_mode=VisionRunningMode.VIDEO, # Setting the running mode to video for frame-by-frame processing
    num_hands=2 # Setting the number of hands to detect (2 in this case)
)


In [ ]:
# ================== ANGLE CALCULATION ==================
def calculate_angle(a, b, c):# Function to calculate the angle between three points (a, b, c)
    a = np.array([a.x, a.y, a.z]) # Converting the points to NumPy arrays for easier calculations
    b = np.array([b.x, b.y, b.z]) 
    c = np.array([c.x, c.y, c.z])

    ba = a - b # Calculating the vectors from point b to points a and c
    bc = c - b

    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc)) # Calculating the cosine of the angle using the dot product and magnitudes of the vectors
    angle = np.arccos(np.clip(cosine_angle, -1.0, 1.0)) # Calculating the angle in radians and clipping the cosine value to avoid numerical issues
    return np.degrees(angle) # Converting the angle to degrees and returning it

In [ ]:
def draw_connections(frame, landmarks, connections, w, h):
    """
    Draws bold light blue lines for bones and red circles for joints.
    
    Parameters:
    - frame: the image to draw on
    - landmarks: list of landmarks
    - connections: list of tuples (start_idx, end_idx)
    - w, h: image width and height
    """
    line_color = (255, 200, 0)  # Light blue in BGR (OpenCV uses BGR)
    line_thickness = 4           # Bold lines

    # Draw connections (bones)
    for start_idx, end_idx in connections:
        start = landmarks[start_idx]
        end = landmarks[end_idx]

        x1, y1 = int(start.x * w), int(start.y * h)
        x2, y2 = int(end.x * w), int(end.y * h)

        cv2.line(frame, (x1, y1), (x2, y2), line_color, line_thickness)

    # Draw joints as red circles
    for lm in landmarks:
        x, y = int(lm.x * w), int(lm.y * h)
        cv2.circle(frame, (x, y), 6, (0, 0, 255), -1)  # Red filled circle

In [ ]:
# ================== CAMERA ==================
cap = cv2.VideoCapture(0)
start_time = time.time()

window_name = "Physio Assessment - Hand + Arm"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

baseline_open_angle = None

with PoseLandmarker.create_from_options(pose_options) as pose_landmarker, \
     HandLandmarker.create_from_options(hand_options) as hand_landmarker:

    while True:

        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        timestamp_ms = int((time.time() - start_time) * 1000)

        # ================== POSE DETECTION ==================
        pose_result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)

        if pose_result.pose_landmarks:
            landmarks = pose_result.pose_landmarks[0]

            draw_connections(frame, landmarks, POSE_CONNECTIONS, w, h, (0,255,0))

            # Shoulder Raise Angle (Left Side Example)
            shoulder_angle = calculate_angle(
                landmarks[23],  # hip
                landmarks[11],  # shoulder
                landmarks[13]   # elbow
            )

            cv2.putText(frame, f"Shoulder Angle: {int(shoulder_angle)} deg",
                        (30, 60),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.9, (255,255,0), 2)

        # ================== HAND DETECTION ==================
        hand_result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

        if hand_result.hand_landmarks:
            for hand_landmarks in hand_result.hand_landmarks:

                draw_connections(frame, hand_landmarks, HAND_CONNECTIONS, w, h)

                # Calculate MCP Angles
                index_angle = calculate_angle(hand_landmarks[0], hand_landmarks[5], hand_landmarks[8])
                middle_angle = calculate_angle(hand_landmarks[0], hand_landmarks[9], hand_landmarks[12])
                ring_angle = calculate_angle(hand_landmarks[0], hand_landmarks[13], hand_landmarks[16])
                pinky_angle = calculate_angle(hand_landmarks[0], hand_landmarks[17], hand_landmarks[20])

                avg_angle = (index_angle + middle_angle + ring_angle + pinky_angle) / 4

                # Convert to percentage
                open_percentage = int(((avg_angle - 60) / (180 - 60)) * 100)
                open_percentage = max(0, min(100, open_percentage))
                closed_percentage = 100 - open_percentage

                # Save baseline when fully open
                if baseline_open_angle is None and open_percentage > 90:
                    baseline_open_angle = avg_angle

                cv2.putText(frame, f"Open: {open_percentage}%",
                            (30, 120),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.9, (0,255,0), 2)

                cv2.putText(frame, f"Closed: {closed_percentage}%",
                            (30, 160),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.9, (0,0,255), 2)

        cv2.imshow(window_name, frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

# ================== CLEANUP ==================
cap.release()
cv2.destroyAllWindows()

TypeError: draw_connections() takes 5 positional arguments but 6 were given

: 